# 01-2. 메시지 구조와 Structured Output

- 예상 시간: 120분
- 선수 실습: 01-1_ai_app_workflow_agent.ipynb
- 실습 난이도: 기초~중급
- 핵심 기술: System/User/Assistant Message, JSON 출력, Pydantic 검증, 모델의 Schema 기반 Structured Output
- 최종 산출물: 검증된 `InputAnalysis` Structured Output
- 버전: 학생용 실습파일 (`TODO`가 표시된 코드 셀을 완성해야 이후 셀이 정상 동작합니다)

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. System, User, Assistant 메시지의 역할을 구분하여 설명할 수 있다.
2. Zero-shot과 Few-shot 프롬프트의 차이를 실행 결과로 비교할 수 있다.
3. Pydantic Schema를 이용해 LLM 출력을 검증할 수 있다.
4. 검증 실패 시 재시도와 기본값 처리 전략을 구현할 수 있다.

## 2. 문제 상황

LLM에게 "이 입력을 분석해줘"라고만 요청하면, 답변은 매번 형식이 달라지는 자유형 텍스트로
돌아온다. 이런 텍스트에서 `input_type`, `keywords`처럼 프로그램이 사용할 값을 정규식 등으로
꺼내려 하면, 문장 표현이 조금만 달라져도 파싱이 깨진다. "JSON으로 출력해줘"라고 요청을
바꿔도 필드가 누락되거나 자료형이 틀린 JSON이 돌아올 수 있어, 이 자체만으로는 안전하지
않다. 이 문제를 해결하려면 출력 형식을 스스로 검증하는 절차가 필요하다.

## 3. 핵심 개념

### 3.1 개념 정의

**System 메시지**는 LLM의 역할, 말투, 제약조건을 지정하는 메시지이고, **User 메시지**는
실제 사용자의 요청이며, **Assistant 메시지**는 LLM이 생성한 응답이다. 세 메시지는 대화
맥락을 구성하는 서로 다른 역할을 담당한다.

**Structured Output**은 미리 정의한 Schema(필드 이름, 자료형, 허용값)를 기준으로 LLM 출력의
형식을 제약하고, 애플리케이션이 그 결과를 Parsing과 Validation을 거쳐 신뢰할 수 있는
데이터로 사용하는 방식이다. 단순히 "JSON으로 출력해줘"라고 요청하는 것과는 다르다.

### 3.2 개념이 필요한 이유

예를 들어 사용자의 입력을 `question`, `document`, `calculation`, `unknown` 중 하나로 분류해서
이후 Routing에 사용해야 하는 상황을 생각해보자. LLM이 "이 입력은 질문인 것 같습니다"처럼
자유형으로 답하면, 애플리케이션은 이 문장에서 정확한 분류값을 안정적으로 추출할 수 없다.
Schema와 검증 절차 없이는 오탈자나 허용되지 않은 값이 그대로 다음 단계로 전달되어
Routing 오류로 이어질 수 있다.

### 3.3 주요 구성요소

| 구성요소 | 역할 |
|---|---|
| Instruction | LLM이 수행할 작업을 지시하는 문장 |
| Context | 작업에 필요한 배경 정보(문서, 이전 대화 등) |
| Example | Few-shot에서 제공하는 입력-출력 예시 |
| Output Format | 출력이 따라야 할 형식(JSON, 필드 이름 등) |
| Schema | 출력 필드와 자료형을 정의한 명세 (Pydantic 모델) |
| Parsing | 문자열 출력을 Python 객체로 변환하는 과정 |
| Validation | 변환된 객체가 Schema 조건을 만족하는지 확인하는 과정 |

### 3.4 동작 과정

```text
System + User 메시지 구성
        ↓
LLM 호출 (Assistant 메시지 생성)
        ↓
문자열 응답을 JSON으로 Parsing
        ↓
Pydantic Schema로 Validation
        ↓
   성공 ──────────────┐
        │              │
   실패 → 재시도 또는 기본값 처리
```

다음 관계를 구분해서 이해해야 한다.

```text
JSON 출력 요청
→ 모델에게 JSON 형식을 지시할 뿐, 형식을 보장하지 않는다.

Structured Output
→ 사전에 정의한 Schema를 기준으로 출력 형식을 제약하고 검증한다.
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `SystemMessage(...)` | System 메시지 |
| `HumanMessage(...)` | User 메시지 |
| `class InputAnalysis(BaseModel)` | 출력 Schema 정의 |
| `InputAnalysis.model_validate_json()` | Parsing + Validation |
| `except ValidationError` | 검증 실패 처리 |

### 3.6 유사 개념과의 차이

| 구분 | JSON 출력 요청 | Structured Output |
|---|---|---|
| 형식 지시 방법 | 프롬프트 문장으로 요청 | Schema로 명시적 정의 |
| 형식 보장 | 보장되지 않음 | 애플리케이션이 검증 |
| 실패 감지 | 어려움(직접 확인 필요) | `ValidationError`로 명확히 감지 |
| 필드 누락 처리 | 알 수 없음 | 검증 단계에서 발견 가능 |

Pydantic은 LLM이 아니라 **애플리케이션 계층**에서 동작하는 검증 기능이다. LLM의 출력
방식을 바꾸는 것이 아니라, 이미 생성된 문자열을 사후에 검증한다는 점에 주의한다.

### 3.7 사용 시점과 적용 조건

사람이 결과를 눈으로만 읽는 단순 요약, 설명처럼 후속 처리가 없는 경우에는 자유형 출력으로
충분하다. 이후 코드가 특정 필드 값을 읽어 분기하거나 저장해야 한다면 반드시 Structured
Output을 사용해야 한다.

### 3.8 한계와 주의사항

- Schema 검증은 형식(자료형, 필수 필드, 허용값)만 확인할 뿐, 내용의 사실 여부는 검증하지
  않는다.
- 검증 실패 시 재시도를 무한정 반복하면 비용과 지연시간이 계속 늘어나므로 재시도 횟수
  제한이 필요하다.
- Schema를 지나치게 세분화하면 LLM이 모든 조건을 만족하는 출력을 생성하기 어려워져
  오히려 실패율이 높아질 수 있다.

### 3.9 자주 발생하는 오해

"모델에게 JSON으로 출력해달라고 했으니 항상 유효한 JSON이 돌아온다"는 오해가 흔하다.
실제로는 코드 블록 기호(` ```json `)가 함께 출력되거나, 필드가 누락되거나, 자료형이
바뀌는 경우가 자주 발생한다. Structured Output은 이런 상황을 막는 것이 아니라,
**감지하고 대응**하기 위한 절차라는 점을 기억해야 한다.

## 4. 실행 구조

이번 실습은 다음 순서로 진행한다.

```text
System 메시지 유무 비교
        ↓
Zero-shot vs Few-shot 비교
        ↓
자유형 출력 → JSON 출력 요청 (형식 불안정성 확인)
        ↓
Pydantic Schema 정의 → 출력 검증
        ↓
ValidationError 처리 → 재시도/기본값 처리
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [ ]:
import json

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import load_jsonl, save_log
from agentic_ai.models import get_chat_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import DATA_DIR, OUTPUT_DIR, PROJECT_ROOT

settings = get_settings()
print_environment_summary(settings, needs_chat_model=True)


## 6. 최소 실행 예제

System 메시지 없이 User 메시지만 보내 기본 호출을 확인한다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

model = get_chat_model()
response = model.invoke([HumanMessage(content="너는 누구야?")])
print("응답 메시지 타입:", type(response).__name__)
print(response.content)

## 7. 단계별 구현

### 7.1 System 메시지 유무 비교

동일한 User 메시지에 System 메시지를 추가했을 때와 추가하지 않았을 때 응답이 어떻게
달라지는지 비교한다.

In [ ]:
user_question = "리모트워크 제도에 대해 설명해줘."

without_system = model.invoke([HumanMessage(content=user_question)])

with_system = model.invoke([
    SystemMessage(content="너는 사내 인사 정책을 안내하는 담당자다. 존댓말로 2문장 이내로 답한다."),
    HumanMessage(content=user_question),
])

print("[System 없음]\n", without_system.content)
print("\n[System 있음]\n", with_system.content)

### 7.2 Zero-shot과 Few-shot 비교

동일한 분류 작업을 예시 없이(Zero-shot) 요청한 경우와 예시를 포함해서(Few-shot) 요청한
경우를 비교한다. 분류 대상은 `question`, `document`, `calculation`, `unknown` 중 하나이다.

In [ ]:
def classify_zero_shot(text: str) -> str:
    """예시 없이 입력 유형을 분류한다."""
    prompt = (
        "다음 입력을 question, document, calculation, unknown 중 하나로 분류해서 "
        "분류명만 출력해줘. 다른 설명은 출력하지 마.\n\n"
        f"입력: {text}"
    )
    response = model.invoke(prompt)
    return response.content.strip()


def classify_few_shot(text: str) -> str:
    """예시를 포함해서 입력 유형을 분류한다."""
    prompt = (
        "다음 예시를 참고해서 입력을 question, document, calculation, unknown 중 "
        "하나로 분류하고 분류명만 출력해줘.\n\n"
        "예시 1) 입력: LangGraph가 뭐야? → question\n"
        "예시 2) 입력: 이 문서를 요약해줘. → document\n"
        "예시 3) 입력: 12 곱하기 8은? → calculation\n\n"
        f"입력: {text}"
    )
    response = model.invoke(prompt)
    return response.content.strip()


test_text = "128을 4로 나누면 얼마야?"
print("zero-shot:", classify_zero_shot(test_text))
print("few-shot :", classify_few_shot(test_text))

### 7.3 자유형 출력의 한계

입력 분석 결과를 자유형 문장으로 요청한 뒤, 여기서 `input_type` 값을 직접 꺼내보면
파싱이 불안정하다는 것을 확인할 수 있다.

In [ ]:
free_form_prompt = "다음 입력을 분석해서 어떤 유형인지, 핵심 내용은 무엇인지 설명해줘.\n\n입력: 이 문서를 요약해줘."
free_form_response = model.invoke(free_form_prompt)
print(free_form_response.content)

# 자유형 텍스트에서 값을 꺼내려는 시도 (안정적이지 않다)
found = "document" in free_form_response.content
print("\n'document'라는 단어가 포함되어 있는가:", found)

### 7.4 JSON 출력 요청

"JSON으로 출력해줘"라고 요청만 바꿔본다. 아래 셀은 Schema 없이 형식만 지시한 경우로,
`json.loads()`가 실패할 수 있다는 점을 관찰하기 위한 것이다.

In [ ]:
json_prompt = (
    "다음 입력을 분석해서 input_type, summary, keywords, confidence 필드를 가진 "
    "JSON으로만 출력해줘.\n\n입력: 이 문서를 요약해줘."
)
json_response = model.invoke(json_prompt)
print(json_response.content)

try:
    parsed = json.loads(json_response.content)
    print("\n파싱 성공:", parsed)
except json.JSONDecodeError as e:
    print("\n파싱 실패:", e)

### 7.5 Pydantic Schema 정의

`src/agentic_ai/schemas.py`의 `InputAnalysis`는 허용값, 범위, 추가 필드 금지와 엄격한
자료형 검사를 함께 정의한다.

```python
class InputAnalysis(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)

    input_type: Literal["question", "document", "calculation", "unknown"]
    summary: str
    keywords: list[str]
    confidence: float = Field(ge=0.0, le=1.0)
```

Pydantic의 기본 설정은 일부 값을 자동 변환하고 추가 필드를 무시할 수 있다. 이 실습은
Routing에 사용할 데이터이므로 `strict=True`, `extra="forbid"`를 사용한다.


In [ ]:
from agentic_ai.schemas import InputAnalysis
from pydantic import ValidationError

print(InputAnalysis.model_fields.keys())

### 7.6 출력 검증

**TODO**: `analyze_input_structured()`를 작성한다. `InputAnalysis`의 필드 이름과 각
필드가 무엇을 의미하는지 프롬프트에 명시하고, `input_type`은 반드시 네 가지 값 중
하나여야 한다고 지시한다. LLM 응답 문자열을 `InputAnalysis.model_validate_json()`으로
검증해서 반환한다.

예상 출력: `InputAnalysis(input_type='document', summary='...', keywords=[...], confidence=0.xx)`

In [ ]:
structured_model = model.with_structured_output(InputAnalysis)


def analyze_input_structured(text: str) -> InputAnalysis:
    """Schema를 모델 호출에 전달하고, 검증된 InputAnalysis를 반환한다."""
    # TODO: 1) SystemMessage에 InputAnalysis의 각 필드(input_type, summary, keywords,
    #       confidence)가 무엇을 의미하는지 명시하고, input_type은 반드시
    #       question/document/calculation/unknown 중 하나여야 한다고 지시한다.
    # TODO: 2) HumanMessage(content=text)와 함께 messages 리스트를 구성한다.
    # TODO: 3) structured_model.invoke(messages)를 호출해 검증된 InputAnalysis를
    #       반환한다.
    raise NotImplementedError("TODO: analyze_input_structured를 완성하세요.")


sample_analysis = analyze_input_structured("이 문서에서 누락된 조항을 찾아줘.")
print(sample_analysis)


### 7.7 ValidationError 처리

의도적으로 잘못된 JSON을 만들어 `ValidationError`가 어떻게 발생하는지 확인한다.
이 셀은 실제 LLM 호출 없이 다음 실패를 결정적으로 재현한다.

- JSON 문법 오류와 필수 필드 누락
- 허용되지 않은 `input_type`과 잘못된 `keywords` 자료형
- 엄격한 Schema에서 금지한 문자열 숫자와 추가 필드


In [ ]:
failure_cases = {
    "JSON 문법 오류": '{"input_type": "question", "summary": "테스트"',
    "confidence 누락": '{"input_type": "question", "summary": "테스트", "keywords": ["a"]}',
    "허용되지 않은 input_type": (
        '{"input_type": "essay", "summary": "테스트", "keywords": ["a"], "confidence": 0.5}'
    ),
    "keywords 자료형 오류": (
        '{"input_type": "question", "summary": "테스트", "keywords": "a,b", "confidence": 0.5}'
    ),
    "confidence 문자열 오류": (
        '{"input_type": "question", "summary": "테스트", "keywords": ["a"], "confidence": "0.5"}'
    ),
    "추가 필드 오류": (
        '{"input_type": "question", "summary": "테스트", "keywords": ["a"], "confidence": 0.5, "extra": 1}'
    ),
}

for case_name, raw_json in failure_cases.items():
    try:
        InputAnalysis.model_validate_json(raw_json)
        print(f"[{case_name}] 검증 통과 (예상과 다름)")
    except (ValidationError, json.JSONDecodeError) as e:
        print(f"[{case_name}] 검증 실패(예상된 결과): {type(e).__name__}")

**결과 해석**: 모든 사례가 Pydantic `ValidationError`로 감지된다.
`model_validate_json()`은 JSON 문법 오류도 `ValidationError`로 감싸서 반환한다.
검증이 없었다면 잘못된 값이 Routing이나 저장 단계로 전달될 수 있다.


## 8. 실행 결과 관찰

`data/samples/sample_inputs.jsonl`의 입력들을 `analyze_input_structured()`로 분석하고
검증 결과를 표로 확인한다.

In [ ]:
sample_inputs = load_jsonl(DATA_DIR / "samples" / "sample_inputs.jsonl")[:4]

observation_rows = []
for item in sample_inputs:
    try:
        analysis = analyze_input_structured(item["text"])
        observation_rows.append(
            {
                "id": item["id"],
                "expected_type": item["expected_type"],
                "actual_type": analysis.input_type,
                "confidence": analysis.confidence,
                "valid": True,
            }
        )
    except ValidationError as e:
        observation_rows.append(
            {"id": item["id"], "expected_type": item["expected_type"], "valid": False, "error": str(e)}
        )

for row in observation_rows:
    print(row)

**결과 해석**: `valid`가 `True`인 항목은 Schema 검증을 통과한 것이고, `actual_type`이
`expected_type`과 일치하는지는 분류 정확도의 문제로 검증 통과 여부와는 별개이다.
검증 통과와 분류 정확도는 서로 다른 기준이라는 점에 주의한다.

## 9. 실패 실험: 재요청 없이 그대로 사용하면?

Schema 검증에 실패했는데도 이를 무시하고 `.model_dump()`처럼 후속 처리를 강행하면
어떤 문제가 생기는지 확인한다.

In [ ]:
broken_json = '{"input_type": "essay", "summary": "테스트", "keywords": ["a"], "confidence": 0.5}'

try:
    bad_analysis = InputAnalysis.model_validate_json(broken_json)
    print("검증 통과:", bad_analysis)
except ValidationError as e:
    print("검증 실패. 원인:")
    for err in e.errors():
        print(" -", err["loc"], err["msg"])

**원인 분석 질문**

- `essay`는 왜 허용되지 않는가? `InputAnalysis`의 `input_type` 정의를 다시 확인한다.
- 이 오류를 무시하고 `bad_analysis.input_type`을 그대로 Routing에 사용했다면 어떤 문제가
  발생했을까?
- 검증 실패를 감지한 뒤 애플리케이션이 취할 수 있는 선택지는 무엇인가?
  (재요청 / 기본값 대체 / 오류로 중단)

## 10. 오류 수정 실습

검증에 실패하면 오류를 그대로 던지지 않고, 안전하게 처리하는 함수를 작성한다.

**TODO**: `safe_analyze_input()`을 작성한다. `analyze_input_structured()`가
`ValidationError`를 던지면, `input_type="unknown"`, `confidence=0.0`인 기본값
`InputAnalysis`를 반환한다.

In [ ]:
def safe_analyze_input(
    text: str,
    analyzer=analyze_input_structured,
    max_attempts: int = 2,
) -> InputAnalysis:
    """출력 검증 실패만 제한 횟수만큼 재시도하고, 모두 실패하면 기본값을 반환한다."""
    if max_attempts < 1:
        raise ValueError("max_attempts는 1 이상이어야 합니다.")
    # TODO: 1) for 반복문으로 최대 max_attempts번까지 analyzer(text)를 호출한다.
    # TODO: 2) ValidationError 또는 ValueError가 발생하면 재시도 횟수를 출력하고
    #       다음 시도로 넘어간다(오류를 그대로 던지지 않는다).
    # TODO: 3) 모든 시도가 실패하면 input_type="unknown", summary="분석 실패",
    #       keywords=[], confidence=0.0인 기본값 InputAnalysis를 반환한다.
    raise NotImplementedError("TODO: safe_analyze_input을 완성하세요.")


result_ok = safe_analyze_input("이 문서를 요약해줘.")
print(result_ok)


**수정 결과 재검증**: `safe_analyze_input()`은 출력 검증 실패만 최대 횟수만큼
재시도한 뒤 `InputAnalysis` 기본값을 반환한다. 인증·네트워크·Rate Limit 같은 운영 오류는
숨기지 않고 호출자에게 전달한다.


In [ ]:
attempt_counter = {"count": 0}


def always_invalid(_: str) -> InputAnalysis:
    attempt_counter["count"] += 1
    raise ValueError("의도적으로 만든 검증 실패")


fallback_result = safe_analyze_input("테스트", analyzer=always_invalid, max_attempts=2)
assert attempt_counter["count"] == 2
assert fallback_result.input_type == "unknown"
assert isinstance(result_ok, InputAnalysis)
print("재시도 및 기본값 재검증 통과")


## 11. 도전 과제

1. `analyze_input_structured()`가 검증에 실패했을 때 기본값을 반환하는 대신, 오류 메시지를
   프롬프트에 포함해 한 번 더 요청하는 재시도 로직을 추가해본다.
2. `InputAnalysis`에 `language: Literal["ko", "en"]` 필드를 추가하고, 기존 검증 실패
   사례들이 여전히 실패하는지 확인해본다.
3. Zero-shot과 Few-shot 분류 결과를 `sample_inputs.jsonl` 전체에 대해 실행하고 일치율을
   비교해본다.

## 12. 테스트

**테스트 유형: 단위 테스트 — 결정적, 외부 API 호출 없음**

Schema 자체의 검증 로직을 LLM 호출 없이 확인한다.

In [ ]:
assert InputAnalysis.model_validate(
    {"input_type": "question", "summary": "s", "keywords": ["a"], "confidence": 0.5}
).input_type == "question"

for raw_json in failure_cases.values():
    try:
        InputAnalysis.model_validate_json(raw_json)
        raise AssertionError("실패해야 할 케이스가 통과했습니다.")
    except (ValidationError, json.JSONDecodeError):
        pass

print("InputAnalysis 검증 테스트 통과")

## 13. 결과 저장

In [ ]:
output_record = {
    "sample_analysis": result_ok.model_dump(),
    "observation_rows": observation_rows,
    "failure_cases_tested": list(failure_cases.keys()),
}
saved_path = save_log(output_record, OUTPUT_DIR / "logs" / "02_structured_output_example.json")
print("저장 위치:", saved_path)

## 14. 핵심 정리

- System, User, Assistant 메시지는 각각 역할 지시, 사용자 요청, 모델 응답이라는
  서로 다른 역할을 담당한다.
- "JSON으로 출력해줘"라는 요청과 Structured Output은 다르다. 후자는 Schema 기반의
  명시적 검증 절차를 포함한다.
- Pydantic 검증은 LLM이 아니라 애플리케이션 계층에서 수행되며, 형식 오류를 실행 중에
  감지할 수 있게 해준다.
- 검증 실패는 무시하지 않고 재시도하거나 안전한 기본값으로 대체해야 한다.

## 15. 확인 문제

1. System 메시지와 User 메시지의 역할 차이를 설명하시오.
2. "JSON으로 출력해줘"라는 프롬프트만으로 Structured Output이 보장되지 않는 이유를
   서술하시오.
3. Pydantic 검증이 LLM 내부 동작이 아니라 애플리케이션 계층의 기능인 이유는 무엇인가?
4. 검증 실패 시 무한 재시도가 바람직하지 않은 이유를 설명하시오.